In [0]:
df = spark.read.format("delta").table( "`ph-com-ai-us-prod-catalog-2553213839599380`.kerendia_oe_journey.final_mdb_oe_2_0_all_instances_v1")

In [0]:
len(df.columns)

In [0]:
df.limit(10).display()

In [0]:
df.limit(10).display()

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import StructType, StructField, IntegerType, LongType
import pandas as pd

# ============================================================
# Expand 7 quarterly instances → 21 monthly instances
# ============================================================
# Photo dates (Instance 1 to 21):
#   1: 5/1/2024-7/31/2024   (start=202405)
#   2: 6/1/2024-8/31/2024   (start=202406)
#   3: 7/1/2024-9/30/2024   (start=202407)
#   ...
#   21: 1/1/2026-3/31/2026  (start=202601)
# ============================================================

# --- Define 21 monthly instances with mapping to nearest quarterly instance ---
# Current quarterly instances: 1=202405, 2=202408, 3=202411, 4=202502, 5=202505, 6=202508, 7=202511

instance_mapping = [
    # (new_instance, new_start_month, new_start_year, old_instance_to_use)
    (1,  202405, 2024, 1),
    (2,  202406, 2024, 1),
    (3,  202407, 2024, 1),
    (4,  202408, 2024, 2),
    (5,  202409, 2024, 2),
    (6,  202410, 2024, 2),
    (7,  202411, 2024, 3),
    (8,  202412, 2024, 3),
    (9,  202501, 2025, 3),
    (10, 202502, 2025, 4),
    (11, 202503, 2025, 4),
    (12, 202504, 2025, 4),
    (13, 202505, 2025, 5),
    (14, 202506, 2025, 5),
    (15, 202507, 2025, 5),
    (16, 202508, 2025, 6),
    (17, 202509, 2025, 6),
    (18, 202510, 2025, 6),
    (19, 202511, 2025, 7),
    (20, 202512, 2025, 7),
    (21, 202601, 2026, 7),
]

# Filter only non-NULL INSTANCE_START_MONTH rows (valid data)
df_valid = df.filter(F.col("INSTANCE_START_MONTH").isNotNull())

print(f"Original df: {df.count()} rows, {len(df.columns)} columns, 7 instances")
print(f"Valid rows (non-NULL INSTANCE_START_MONTH): {df_valid.count()}")

# --- Build expanded df: for each new monthly instance, get data from mapped quarterly instance ---
expanded_dfs = []

for new_inst, new_start_month, new_start_year, old_inst in instance_mapping:
    # Get data from the corresponding old quarterly instance
    inst_df = df_valid.filter(F.col("INSTANCE") == old_inst)
    
    # Replace INSTANCE, INSTANCE_START_MONTH, INSTANCE_START_YEAR with new values
    inst_df = inst_df \
        .withColumn("INSTANCE", F.lit(new_inst)) \
        .withColumn("INSTANCE_START_MONTH", F.lit(new_start_month)) \
        .withColumn("INSTANCE_START_YEAR", F.lit(new_start_year))
    
    expanded_dfs.append(inst_df)

# Union all 21 instance DataFrames
df_21 = expanded_dfs[0]
for i in range(1, len(expanded_dfs)):
    df_21 = df_21.unionByName(expanded_dfs[i])

print(f"\nExpanded df: {df_21.count()} rows, {len(df_21.columns)} columns, 21 instances")
print(f"\nInstance distribution:")
df_21.groupBy("INSTANCE", "INSTANCE_START_MONTH").count().orderBy("INSTANCE").show(21, truncate=False)

# Reassign df
df = df_21
print("\n✅ df now has 21 monthly instances!")

In [0]:
from pyspark.sql import functions as F
print(f"Distinct INSTANCE values: {df.select('INSTANCE').distinct().count()}")
print(f"Rows: {df.count()}, Columns: {len(df.columns)}")